In [1]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Same loading and cleaning as the EDA notebook
df = pd.read_csv("../data/spam.csv", encoding="latin-1")[["v1", "v2"]]
df.columns = ["label", "message"]
df = df.drop_duplicates().reset_index(drop=True)
df["target"] = df["label"].map({"ham": 0, "spam": 1})

In [2]:
# Basic text cleaning
def clean_text(text):
    text = text.lower()                          # "FREE" and "free" become the same word
    text = re.sub(r"[^a-z0-9\s]", " ", text)     # replace punctuation and symbols with a space
    text = re.sub(r"\s+", " ", text).strip()     # collapse extra spaces
    return text

df["clean"] = df["message"].apply(clean_text)
print(df[["message", "clean"]].head())

                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                               clean  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in 2 a wkly comp to win fa cup fina...  
3        u dun say so early hor u c already then say  
4  nah i don t think he goes to usf he lives arou...  


In [3]:
# Split BEFORE converting text to numbers
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["target"],
    test_size=0.2,
    random_state=42,
    stratify=df["target"],   # keeps the same spam/ham ratio in both sets
)
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())   # both should be close to 0.126

(4135,) (1034,)
0.12623941958887544 0.12669245647969052


In [4]:
# Convert text to TF-IDF numbers
vectorizer = TfidfVectorizer(stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)   # learn the vocabulary from training data only
X_test_tfidf = vectorizer.transform(X_test)         # apply it to test data, don't relearn

print(X_train_tfidf.shape)   # (number of messages, number of unique words)

(4135, 7285)


In [5]:
# Train the first model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

preds = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, preds))

# Try it on your own messages
samples = [
    "Congratulations! You have won a free prize, call now to claim",
    "Hey, are we meeting for lunch tomorrow?",
]
print(model.predict(vectorizer.transform([clean_text(s) for s in samples])))  # 1 = spam, 0 = ham

Accuracy: 0.9661508704061895
[1 0]
